# 02 — LangChain Models, Messages & Structured Output

## Learning requirements
- dùng provider-independent chat model;
- hiểu `HumanMessage`, `AIMessage`, `SystemMessage`, `ToolMessage`;
- inspect metadata/content blocks thay vì chỉ print final text;
- dùng Pydantic cho typed output;
- hiểu provider-native structured output vs tool strategy.

LangChain v1 lấy model/messages/tools/agent làm core abstractions. Đừng xây learning path mới quanh `LLMChain`.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
while not (root / "requirements.txt").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

from src.providers import get_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

model = get_chat_model()
messages = [
    SystemMessage("You are a concise software architecture tutor."),
    HumanMessage("What is the difference between a workflow and an agent?")
]
response = model.invoke(messages)

print("TYPE:", type(response).__name__)
print("CONTENT:", response.content)
print("TEXT:", getattr(response, "text", None))
print("USAGE:", getattr(response, "usage_metadata", None))

## Message lifecycle

Bạn cần nhìn full object vì modern providers có thể trả:
- text blocks,
- reasoning metadata,
- tool calls,
- citations,
- usage metadata,
- multimodal blocks.

Không assume `response.content` luôn chỉ là một string.

In [ ]:
# Streaming
for chunk in model.stream("Explain context engineering in about 3 sentences."):
    text = getattr(chunk, "text", "")
    if text:
        print(text, end="", flush=True)

## Structured output

Free-text phù hợp cho conversation. Business process thường cần typed data.

Ví dụ interview/CV processing cần object có schema cố định thay vì parse prose.

In [ ]:
from pydantic import BaseModel, Field

class CandidateProfile(BaseModel):
    name: str = Field(description="Candidate full name")
    skills: list[str]
    experience_years: float = Field(ge=0)
    seniority: str

structured_model = model.with_structured_output(CandidateProfile)

candidate = structured_model.invoke(
    "Nguyen Van A is a Python and React developer with 2.5 years of professional experience. "
    "Treat under 3 years as junior for this exercise."
)

print(candidate)
print(type(candidate))

## Exercise

Tạo 3 schema:
1. `ProjectRequirement`
2. `InterviewQuestion`
3. `RiskAssessment`

Test bằng input cố tình thiếu field hoặc ambiguous. Ghi lại model/provider xử lý ra sao.

## Required output
`CandidateProfile` hợp lệ + observation table gồm:
- provider,
- model,
- schema success/failure,
- latency,
- notes.

## Done criteria
- Không parse JSON bằng regex nếu provider/LangChain đã có structured output.
- Biết khi nào cần Pydantic validation.
- Biết content và content blocks không phải lúc nào cùng shape.